In [ ]:
# Cell 1: Check TPU, RAM, disk and Internet, install PyTorch/XLA, run a TPU smoke test
# Kaggle：右側 Session options > Accelerator 選 "TPU v5e-8"，並開啟 Internet。
import os, platform, shutil, socket, subprocess, sys, time

SESSION_START = time.time()
print(f"Python: {platform.python_version()}")

def sh(cmd):
    print("$", cmd, flush=True)
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if res.returncode != 0:
        print(res.stdout[-3000:], res.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return res.stdout

if not (os.environ.get("TPU_ACCELERATOR_TYPE") or os.path.exists("/dev/vfio") or any(n.startswith("accel") for n in os.listdir("/dev"))):
    raise RuntimeError("找不到 TPU，請於右側 Session options > Accelerator 選擇 TPU v5e-8。")
print(f"TPU: {os.environ.get('TPU_ACCELERATOR_TYPE', '(偵測到 TPU 裝置)')}")

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
print(f"RAM: {ram_gb:.0f} GB")
if ram_gb < 80:
    print("⚠ 主機記憶體不足 80GB，載入 58GB 的 bf16 模型可能會失敗。")

# 模型約 58GB：挑剩餘空間最大的位置放 Hugging Face 快取
candidates = ["/tmp", "/kaggle/tmp"]
for d in candidates:
    os.makedirs(d, exist_ok=True)
WORK = max(candidates, key=lambda d: shutil.disk_usage(d).free)
free_gb = shutil.disk_usage(WORK).free / 1024**3
print(f"模型快取位置：{WORK}（剩餘 {free_gb:.0f} GB）")
if free_gb < 75:
    raise RuntimeError(f"磁碟空間不足：需要約 75GB，{WORK} 只剩 {free_gb:.0f}GB。")

try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
    print("Internet: OK")
except OSError:
    raise RuntimeError("無網路連線，請於右側 Session options 開啟 Internet on。")

# PyTorch/XLA 2.8 是社群在 Kaggle v5e-8 上確認可用的版本（2.9 在 v5e 上載入大模型會 bad_alloc）
TORCH_PINS = ["torch==2.8.0", "torchvision==0.23.0", "torch_xla[tpu]==2.8.0"]
have = subprocess.run([sys.executable, "-c", "import torch, torch_xla; print(torch.__version__, torch_xla.__version__)"],
                      capture_output=True, text=True).stdout.split()
if len(have) != 2 or not all(v.startswith("2.8.0") for v in have):
    sh(f"{sys.executable} -m pip install -q " + " ".join(f"'{p}'" for p in TORCH_PINS)
       + " -f https://storage.googleapis.com/libtpu-releases/index.html")

# TPU 同一時間只能被一個程序使用：smoke test 在子程序跑，notebook 本身不碰 TPU
SMOKE = r"""
import numpy as np, torch, torch_xla, torch_xla.runtime as xr, torch_xla.distributed.spmd as xs
xr.use_spmd()
n = xr.global_runtime_device_count()
mesh = xs.Mesh(np.arange(n), (n,), ("model",))
dev = torch_xla.device()
a = torch.randn(2048, 2048, dtype=torch.bfloat16)
b = torch.randn(2048, 2048, dtype=torch.bfloat16)
xa, xb = a.to(dev), b.to(dev)
xs.mark_sharding(xb, mesh, (None, "model"))
c = (xa @ xb).cpu().float()
ref = a.float() @ b.float()
rel = ((c - ref).abs().max() / ref.abs().max()).item()
print(f"TPU_OK devices={n} type={xr.device_type()} rel_err={rel:.4f}")
assert n >= 8 and rel < 0.05, (n, rel)
"""
res = subprocess.run([sys.executable, "-c", SMOKE], text=True, capture_output=True,
                     env=dict(os.environ, PJRT_DEVICE="TPU"), timeout=15 * 60)
print(res.stdout.strip())
if res.returncode != 0 or "TPU_OK" not in res.stdout:
    print(res.stderr[-4000:])
    raise RuntimeError("TPU smoke test 失敗：PyTorch/XLA 無法在這台 TPU 上執行，請看上方錯誤訊息。")
print("TPU smoke test 通過。")

In [ ]:
# Cell 2: Download Drive folder & match reference images by item name
IN_FOLDER_URL = ""  # 貼上您的 Google Drive 共用資料夾網址；留空則使用內建測試 prompts.json

import glob, json, os, shutil, subprocess, sys

ROOT = "/kaggle/working/imagegen"
INPUT_DIR = os.path.join(ROOT, "in")
DEBUG_DIR = os.path.join(ROOT, "debug")
REF_DIR = os.path.join(WORK, "prompts_in")
PROMPTS_PATH = os.path.join(ROOT, "prompts.json")
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(REF_DIR, exist_ok=True)

sources = []
if IN_FOLDER_URL.strip():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown==5.2.0"], check=True)
    import gdown
    shutil.rmtree(REF_DIR, ignore_errors=True)
    os.makedirs(REF_DIR, exist_ok=True)
    gdown.download_folder(IN_FOLDER_URL.strip(), output=REF_DIR, quiet=False, remaining_ok=True)
    sources.append(REF_DIR)
sources.append("/kaggle/input")

found_prompts = [p for s in sources for p in glob.glob(os.path.join(s, "**", "prompts.json"), recursive=True)]
if found_prompts:
    shutil.copy(found_prompts[0], PROMPTS_PATH)
    print(f"已載入外部設定檔：{found_prompts[0]}")
else:
    builtin_prompts = {
        # lightning: 0 = 完整 40 步（品質最好）；8 或 4 = Lightning LoRA 快速版
        # two_stage: true = 先換姿勢、再換服裝，身材較不易跑掉，但時間加倍
        "defaults": {"seed": 42, "pose": "a-pose", "steps": 40, "cfg": 4.0, "lightning": 0, "two_stage": False,
                     "style": "photorealistic, natural skin texture"},
        "items": [
            {
                "name": "character-tifa",
                "outfit": "futuristic silver and black cybernetic tactical armor, combat shorts, knee-high boots",
                "pose": "a-pose",
                "seed": 1001
            },
            {
                "name": "character-mage_male",
                "outfit": "dark red and gold linen robes, leather bracers",
                "body": "30yo Asian male battle mage, short black hair, lean athletic build",
                "pose": "a-pose",
                "seed": 777
            }
        ]
    }
    with open(PROMPTS_PATH, "w", encoding="utf-8") as f:
        json.dump(builtin_prompts, f, indent=2, ensure_ascii=False)
    print("未提供外部 prompts.json，已建立內建測試設定檔。")

IMG_EXTS = (".png", ".jpg", ".jpeg", ".webp")
ref_images_map = {}
for s in sources:
    for root_dir, _, files in os.walk(s):
        for file in files:
            if file.lower().endswith(IMG_EXTS):
                stem = os.path.splitext(file)[0]
                ref_images_map[stem] = os.path.join(root_dir, file)

def find_ref(name):
    short_name = name.split("-", 1)[1] if "-" in name else name
    return ref_images_map.get(name) or ref_images_map.get(short_name) or ""

with open(PROMPTS_PATH, "r", encoding="utf-8") as f:
    prompts_data = json.load(f)

defaults = prompts_data.get("defaults", {})
todo = [item for item in prompts_data.get("items", []) if not item.get("done")]
print(f"\n共 {len(prompts_data.get('items', []))} 筆項目，待處理 {len(todo)} 筆：")
for item in todo:
    name = item["name"]
    if not (item.get("outfit") or item.get("prompt", "")).strip():
        raise ValueError(f"{name}：缺少 outfit（或舊版 prompt）服裝描述。")
    item["_ref_image"] = find_ref(name)
    pose_name = item.get("pose", defaults.get("pose", "a-pose"))
    mode = "兩階段" if item.get("two_stage", defaults.get("two_stage", False)) else "單階段"
    if item["_ref_image"]:
        print(f"  - {name}: [參考圖: {os.path.basename(item['_ref_image'])}] | 姿勢={pose_name} | {mode}")
    else:
        print(f"  - {name}: [純文字模式] | 姿勢={pose_name}")

with open(PROMPTS_PATH, "w", encoding="utf-8") as f:
    json.dump(prompts_data, f, indent=2, ensure_ascii=False)

In [ ]:
# Cell 3: Install dependencies, download Qwen-Image-Edit-2511, and write the TPU worker script
import os, subprocess, sys, time

MARK = os.path.join(WORK, ".qwen_edit_tpu_v1")
CONSTRAINTS = os.path.join(WORK, "constraints.txt")
HF_CACHE = os.path.join(WORK, "hf_cache")
REMBG_CACHE = os.path.join(WORK, "rembg_cache")
XLA_CACHE = os.path.join(WORK, "xla_cache")
for d in (HF_CACHE, REMBG_CACHE, XLA_CACHE):
    os.makedirs(d, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE

PINNED = [
    "diffusers==0.41.0", "transformers==5.19.0", "huggingface_hub==1.33.0", "accelerate==1.15.0",
    "peft==0.21.2", "safetensors>=0.8.0", "onnxruntime==1.22.1", "rembg==2.0.67",
    "opencv-python-headless", "hf_xet"
]

if not os.path.exists(MARK):
    with open(CONSTRAINTS, "w") as f:
        f.write("torch==2.8.0\ntorchvision==0.23.0\ntorch_xla==2.8.0\n")
    sh(f"{sys.executable} -m pip install -q -c {CONSTRAINTS} " + " ".join(f"'{p}'" for p in PINNED))
    open(MARK, "w").close()

lightning = int(prompts_data.get("defaults", {}).get("lightning", 0))
code = f'''
import time
from huggingface_hub import snapshot_download, hf_hub_download
t0 = time.time()
print("正在下載 Qwen-Image-Edit-2511（約 58GB）...", flush=True)
snapshot_download("Qwen/Qwen-Image-Edit-2511")
if {lightning} in (4, 8):
    hf_hub_download("lightx2v/Qwen-Image-Edit-2511-Lightning", "Qwen-Image-Edit-2511-Lightning-{lightning}steps-V1.0-bf16.safetensors")
print(f"模型準備完成，耗時 {{(time.time() - t0) / 60:.1f}} 分鐘。", flush=True)
'''
subprocess.run([sys.executable, "-c", code], check=True, env=dict(os.environ, HF_HOME=HF_CACHE))

worker_code = '''import argparse, json, math, os, time, cv2, numpy as np, torch
from PIL import Image
from rembg import new_session, remove

import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
import torch_xla.distributed.spmd as xs
from diffusers import QwenImageEditPlusPipeline, FlowMatchEulerDiscreteScheduler
from diffusers.pipelines.qwenimage.pipeline_qwenimage_edit_plus import calculate_dimensions, CONDITION_IMAGE_SIZE

MODEL_ID = "Qwen/Qwen-Image-Edit-2511"
LIGHTNING_REPO = "lightx2v/Qwen-Image-Edit-2511-Lightning"
W, H = 832, 1216      # 固定輸出解析度：所有形狀不變，XLA 只需編譯一次
TEXT_BUCKET = 256     # 文字向量補齊到 256 的倍數，不同長度的描述共用同一張編譯圖

# ==============================================================================
# 【姿勢設定 (POSE_PRESETS)】與 2×T4 版相同：數值為相對畫布寬 (w)、高 (h) 的比例
#   - arm_dx1, arm_dy1: 手肘距離中心線的水平距離與高度
#   - arm_dx2, arm_dy2: 手腕距離中心線的水平距離與高度
#   - leg_dx          : 膝蓋與腳踝距離中心線的水平距離
# POSE_TEXT 是同一個姿勢的文字描述，和骨架圖一起給模型，模型不理骨架時仍有文字可依。
# ==============================================================================
POSE_PRESETS = {
    "a-pose":        {"arm_dx1": 0.19, "arm_dy1": 0.33, "arm_dx2": 0.26, "arm_dy2": 0.46, "leg_dx": 0.08},
    "wide-a-pose":   {"arm_dx1": 0.23, "arm_dy1": 0.29, "arm_dx2": 0.34, "arm_dy2": 0.38, "leg_dx": 0.10},
    "t-pose":        {"arm_dx1": 0.24, "arm_dy1": 0.20, "arm_dx2": 0.38, "arm_dy2": 0.20, "leg_dx": 0.08},
    "straight":      {"arm_dx1": 0.14, "arm_dy1": 0.34, "arm_dx2": 0.16, "arm_dy2": 0.48, "leg_dx": 0.06},
    "hands-on-hips": {"arm_dx1": 0.24, "arm_dy1": 0.32, "arm_dx2": 0.11, "arm_dy2": 0.43, "leg_dx": 0.09},
}
POSE_TEXT = {
    "a-pose": "A-pose: arms straight and angled about 35 degrees down and away from the body, palms facing the thighs, feet shoulder-width apart",
    "wide-a-pose": "wide A-pose: arms straight and raised about 55 degrees away from the body, feet slightly wider than shoulders",
    "t-pose": "T-pose: both arms stretched straight out horizontally at shoulder height, palms down, feet shoulder-width apart",
    "straight": "relaxed straight standing pose: arms hanging straight down beside the body, feet slightly apart",
    "hands-on-hips": "standing with both hands on the hips, elbows pointing outward, feet shoulder-width apart",
}
NEGATIVE = (
    "skeleton lines, colored stick figure, cropped, out of frame, head cut off, feet cut off, multiple people, "
    "side view, back view, turned head, extra limbs, extra fingers, deformed hands, bad anatomy, "
    "different face, different body shape, blurry, lowres, text, watermark, busy background, shadows on background"
)

def make_openpose_map(pose_name="a-pose", w=W, h=H):
    """根據 POSE_PRESETS 繪製對稱全身 OpenPose 骨架圖"""
    p = POSE_PRESETS.get(pose_name.lower(), POSE_PRESETS["a-pose"])
    canvas = np.zeros((h, w, 3), dtype=np.uint8)
    cx = w // 2
    pts = {
        0:  (cx, int(h * 0.115)),
        1:  (cx, int(h * 0.190)),
        2:  (cx - int(w * 0.11), int(h * 0.20)),
        3:  (cx - int(w * p["arm_dx1"]), int(h * p["arm_dy1"])),
        4:  (cx - int(w * p["arm_dx2"]), int(h * p["arm_dy2"])),
        5:  (cx + int(w * 0.11), int(h * 0.20)),
        6:  (cx + int(w * p["arm_dx1"]), int(h * p["arm_dy1"])),
        7:  (cx + int(w * p["arm_dx2"]), int(h * p["arm_dy2"])),
        8:  (cx - int(w * 0.07), int(h * 0.47)),
        9:  (cx - int(w * p["leg_dx"]), int(h * 0.68)),
        10: (cx - int(w * p["leg_dx"]), int(h * 0.89)),
        11: (cx + int(w * 0.07), int(h * 0.47)),
        12: (cx + int(w * p["leg_dx"]), int(h * 0.68)),
        13: (cx + int(w * p["leg_dx"]), int(h * 0.89)),
        14: (cx - int(w * 0.03), int(h * 0.10)),
        15: (cx + int(w * 0.03), int(h * 0.10)),
        16: (cx - int(w * 0.06), int(h * 0.11)),
        17: (cx + int(w * 0.06), int(h * 0.11)),
    }
    limbs = [
        (1, 2, (255, 0, 0)), (1, 5, (255, 85, 0)), (2, 3, (255, 170, 0)), (3, 4, (255, 255, 0)),
        (5, 6, (170, 255, 0)), (6, 7, (85, 255, 0)), (1, 8, (0, 255, 0)), (8, 9, (0, 255, 85)),
        (9, 10, (0, 255, 170)), (1, 11, (0, 255, 255)), (11, 12, (0, 170, 255)), (12, 13, (0, 85, 255)),
        (1, 0, (0, 0, 255)), (0, 14, (255, 0, 170)), (14, 16, (255, 0, 255)), (0, 15, (170, 0, 255)), (15, 17, (255, 0, 85))
    ]
    for i1, i2, col in limbs:
        p1, p2 = pts[i1], pts[i2]
        mx, my = (p1[0] + p2[0]) // 2, (p1[1] + p2[1]) // 2
        length = math.hypot(p1[0] - p2[0], p1[1] - p2[1])
        angle = math.degrees(math.atan2(p2[1] - p1[1], p2[0] - p1[0]))
        poly = cv2.ellipse2Poly((mx, my), (int(length / 2), 6), int(angle), 0, 360, 1)
        cv2.fillConvexPoly(canvas, poly, col)
    for pt in pts.values():
        cv2.circle(canvas, pt, 6, (255, 255, 255), thickness=-1)
    return Image.fromarray(canvas)

def load_rgb(path):
    img = Image.open(path)
    if img.mode == "RGBA":
        bg = Image.new("RGB", img.size, (128, 128, 128))
        bg.paste(img, mask=img.split()[3])
        return bg
    return img.convert("RGB")

def normalize_ref(path, rembg_session):
    """參考圖去背、裁到人物範圍，置中貼到 W×H 灰底：模型只看人物，輸入形狀也固定"""
    rgba = remove(load_rgb(path), session=rembg_session)
    box = rgba.getchannel("A").point(lambda a: 255 if a > 16 else 0).getbbox()
    if box:
        rgba = rgba.crop(box)
    scale = min(W * 0.9 / rgba.width, H * 0.92 / rgba.height)
    rgba = rgba.resize((max(1, int(rgba.width * scale)), max(1, int(rgba.height * scale))), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (W, H), (128, 128, 128))
    canvas.paste(rgba, ((W - rgba.width) // 2, (H - rgba.height) // 2), rgba)
    return canvas

def pad_to_square_rgba(rgba_img, target_size=1024):
    rgba_img.thumbnail((target_size, target_size), Image.Resampling.LANCZOS)
    canvas = Image.new("RGBA", (target_size, target_size), (0, 0, 0, 0))
    canvas.paste(rgba_img, ((target_size - rgba_img.width) // 2, (target_size - rgba_img.height) // 2), rgba_img)
    return canvas

def shard_transformer(transformer, mesh, device):
    """
    20B transformer 以張量平行切到 8 顆 TPU（Megatron 式）：
    q/k/v 與 FFN 第一層切輸出維度，輸出投影與 FFN 第二層切輸入維度，其餘小參數複製到每顆。
    逐層搬移，避免整個模型先完整複製到每顆 16GB 的晶片上。
    """
    n = mesh.size()
    row_parallel = ("to_out.0", "to_add_out", "net.2")
    count = 0
    for name, module in transformer.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        out_f, in_f = module.weight.shape
        if name.endswith(row_parallel) and in_f % n == 0:
            spec = (None, "model")
            bias_spec = None
        elif out_f % n == 0:
            spec = ("model", None)
            bias_spec = ("model",)
        else:
            continue
        module.weight = torch.nn.Parameter(module.weight.data.to(device), requires_grad=False)
        xs.mark_sharding(module.weight, mesh, spec)
        if module.bias is not None:
            module.bias = torch.nn.Parameter(module.bias.data.to(device), requires_grad=False)
            if bias_spec:
                xs.mark_sharding(module.bias, mesh, bias_spec)
        count += 1
        if count % 40 == 0:
            xm.mark_step()
    transformer.to(device)
    xm.mark_step()
    print(f"transformer 已切分 {count} 個 Linear 到 {n} 顆 TPU", flush=True)

def condition_images(pipe, images):
    """與 pipeline 內部相同的縮圖方式，讓 CPU 上算的文字向量對得上 TPU 上的圖"""
    out = []
    for img in images:
        cw, ch = calculate_dimensions(CONDITION_IMAGE_SIZE, img.width / img.height)
        out.append(pipe.image_processor.resize(img, ch, cw))
    return out

def encode(pipe, prompt, images, device):
    """Qwen2.5-VL 在 CPU 上編碼（每筆只跑一次），回傳未補齊的向量與遮罩"""
    with torch.no_grad():
        embeds, mask = pipe.encode_prompt(prompt, image=condition_images(pipe, images), device=torch.device("cpu"))
    if mask is None:
        mask = torch.ones(embeds.shape[:2], dtype=torch.long)
    return embeds, mask

def pad_pair(pos, neg, device):
    """正、負向量補齊到同一個 TEXT_BUCKET 倍數長度；補的位置遮罩為 0，不影響結果"""
    longest = max(pos[0].shape[1], neg[0].shape[1] if neg else 0)
    target = math.ceil(longest / TEXT_BUCKET) * TEXT_BUCKET
    def pad(pair):
        embeds, mask = pair
        extra = target - embeds.shape[1]
        embeds = torch.cat([embeds, embeds.new_zeros(embeds.shape[0], extra, embeds.shape[2])], dim=1)
        mask = torch.cat([mask, mask.new_zeros(mask.shape[0], extra)], dim=1)
        return embeds.to(device), mask.to(device)
    return pad(pos), (pad(neg) if neg else (None, None))

def edit(pipe, device, images, prompt, steps, cfg, seed):
    t0 = time.time()
    use_cfg = cfg > 1.0
    pos = encode(pipe, prompt, images, device)
    neg = encode(pipe, NEGATIVE, images, device) if use_cfg else None
    (pe, pm), (ne, nm) = pad_pair(pos, neg, device)
    t1 = time.time()
    out = pipe(
        image=images,
        prompt_embeds=pe, prompt_embeds_mask=pm,
        negative_prompt_embeds=ne, negative_prompt_embeds_mask=nm,
        true_cfg_scale=cfg, num_inference_steps=steps,
        width=W, height=H,
        generator=torch.Generator("cpu").manual_seed(seed),
    ).images[0]
    print(f"    文字編碼 {t1 - t0:.0f}s，生成 {time.time() - t1:.0f}s（含首次編譯）", flush=True)
    return out

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompts", required=True)
    parser.add_argument("--out_dir", required=True)
    parser.add_argument("--debug_dir", required=True)
    parser.add_argument("--xla_cache", required=True)
    args = parser.parse_args()

    with open(args.prompts, "r", encoding="utf-8") as f:
        data = json.load(f)
    defaults = data.get("defaults", {})
    todo = [item for item in data.get("items", []) if not item.get("done")]
    if not todo:
        print("沒有待處理項目。", flush=True)
        return
    os.makedirs(args.out_dir, exist_ok=True)
    os.makedirs(args.debug_dir, exist_ok=True)

    xr.initialize_cache(args.xla_cache, readonly=False)
    xr.use_spmd()
    n = xr.global_runtime_device_count()
    mesh = xs.Mesh(np.arange(n), (n,), ("model",))
    device = torch_xla.device()

    t0 = time.time()
    pipe = QwenImageEditPlusPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
    lightning = int(defaults.get("lightning", 0))
    if lightning in (4, 8):
        pipe.load_lora_weights(LIGHTNING_REPO, weight_name=f"Qwen-Image-Edit-2511-Lightning-{lightning}steps-V1.0-bf16.safetensors")
        pipe.fuse_lora(lora_scale=1.0)
        pipe.unload_lora_weights()
        pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config({
            "base_image_seq_len": 256, "base_shift": math.log(3), "invert_sigmas": False,
            "max_image_seq_len": 8192, "max_shift": math.log(3), "num_train_timesteps": 1000,
            "shift": 1.0, "shift_terminal": None, "stochastic_sampling": False,
            "time_shift_type": "exponential", "use_beta_sigmas": False, "use_dynamic_shifting": True,
            "use_exponential_sigmas": False, "use_karras_sigmas": False,
        })
        print(f"已融合 Lightning {lightning} 步 LoRA", flush=True)
    for module in (pipe.transformer, pipe.vae, pipe.text_encoder):
        module.requires_grad_(False).eval()
    shard_transformer(pipe.transformer, mesh, device)
    pipe.vae.to(device)          # VAE 很小，每顆各放一份
    # text encoder 留在 CPU：每筆只編碼一次，也避免文字長度變動造成 TPU 重新編譯
    assert pipe._execution_device.type == "xla", pipe._execution_device
    rembg_session = new_session("isnet-general-use")
    print(f"model ready（{(time.time() - t0) / 60:.1f} 分鐘）", flush=True)

    keep_default = defaults.get("keep", "face, facial features, hairstyle, hair color, body shape, body proportions, height and skin tone")
    style_default = defaults.get("style", "photorealistic, natural skin texture")
    for item in todo:
        name = item["name"]
        out_png = os.path.join(args.out_dir, f"{name}-front.png")
        seed = int(item.get("seed", defaults.get("seed", 42)))
        pose = item.get("pose", defaults.get("pose", "a-pose")).lower()
        pose_text = POSE_TEXT.get(pose, POSE_TEXT["a-pose"])
        outfit = (item.get("outfit") or item.get("prompt", "")).strip().rstrip(".")
        keep = item.get("keep", keep_default)
        style = item.get("style", style_default)
        body = item.get("body", "")
        steps = lightning or int(item.get("steps", defaults.get("steps", 40)))
        cfg = 1.0 if lightning else float(item.get("cfg", defaults.get("cfg", 4.0)))
        two_stage = bool(item.get("two_stage", defaults.get("two_stage", False)))
        ref_path = item.get("_ref_image", "")
        skeleton = make_openpose_map(pose)
        finish = (f"Full body from head to feet, straight-on front view, symmetrical, looking straight at the camera, "
                  f"centered, even soft studio lighting, plain solid neutral grey background. {style}.")
        body_note = f" Body: {body}." if body else ""

        try:
            if ref_path and os.path.exists(ref_path):
                ref = normalize_ref(ref_path, rembg_session)
                ref.save(os.path.join(args.debug_dir, f"{name}-ref.png"))
                if two_stage:
                    print(f"[{name}] 第 1 階段：只換姿勢（{pose}）", flush=True)
                    posed = edit(pipe, device, [ref, skeleton],
                                 f"Picture 1 shows a person. Picture 2 is a colored pose skeleton. Redraw the same person "
                                 f"from picture 1 with the same clothing, keeping the {keep} exactly the same, but standing "
                                 f"in the exact pose of the skeleton in picture 2: {pose_text}.{body_note} {finish} "
                                 f"Do not draw the skeleton.", steps, cfg, seed)
                    posed.save(os.path.join(args.debug_dir, f"{name}-posed.png"))
                    print(f"[{name}] 第 2 階段：只換服裝", flush=True)
                    img = edit(pipe, device, [posed],
                               f"Change only the clothing of the person in picture 1: now wearing {outfit}. Keep the {keep}, "
                               f"the pose, the framing and the grey background exactly the same. {style}.",
                               steps, cfg, seed)
                else:
                    print(f"[{name}] 參考圖編輯：換服裝與姿勢（{pose}）", flush=True)
                    img = edit(pipe, device, [ref, skeleton],
                               f"Picture 1 shows the character. Picture 2 is a colored pose skeleton of the target pose. "
                               f"Draw the same person from picture 1, keeping the {keep} exactly the same, now wearing "
                               f"{outfit}, standing in the exact pose of the skeleton in picture 2: {pose_text}.{body_note} "
                               f"{finish} Do not draw the skeleton.", steps, cfg, seed)
            else:
                print(f"[{name}] 沒有參考圖：純文字生成（{pose}）", flush=True)
                blank = Image.new("RGB", (W, H), (128, 128, 128))
                img = edit(pipe, device, [blank, skeleton],
                           f"Picture 1 is an empty grey canvas. Picture 2 is a colored pose skeleton. On the grey canvas, "
                           f"draw {body or 'a person with a natural, well-proportioned body'}, wearing {outfit}, standing "
                           f"in the exact pose of the skeleton in picture 2: {pose_text}. {finish} Do not draw the skeleton.",
                           steps, cfg, seed)
        except Exception as error:
            print(f"[{name}] 失敗：{type(error).__name__}: {error}", flush=True)
            continue

        img.save(os.path.join(args.debug_dir, f"{name}-raw.png"))
        rgba = remove(img, session=rembg_session)
        pad_to_square_rgba(rgba, target_size=1024).save(out_png)
        print(f"[{name}] 完成並已儲存：{out_png}", flush=True)

if __name__ == "__main__":
    main()
'''

WORKER_PATH = os.path.join(WORK, "run_qwen_edit_tpu.py")
with open(WORKER_PATH, "w", encoding="utf-8") as f:
    f.write(worker_code)
print(f"TPU 生圖核心 {WORKER_PATH} 寫入完成。")

In [ ]:
# Cell 4: Run generation on the TPU (one process uses all 8 chips) & pack into ZIP
import json, os, subprocess, sys, time, zipfile

env = dict(
    os.environ,
    PJRT_DEVICE="TPU",
    XLA_USE_SPMD="1",
    HF_HOME=HF_CACHE,
    U2NET_HOME=REMBG_CACHE,
    TOKENIZERS_PARALLELISM="false",
)
cmd = [
    sys.executable, "-u", WORKER_PATH,
    "--prompts", PROMPTS_PATH,
    "--out_dir", INPUT_DIR,
    "--debug_dir", DEBUG_DIR,
    "--xla_cache", XLA_CACHE,
]
t0 = time.time()
proc = subprocess.Popen(cmd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in proc.stdout:
    if "it/s]" in line or "s/it]" in line or "B/s]" in line:
        continue
    print(line, end="", flush=True)
proc.wait()
print(f"\n生成程序結束（exit {proc.returncode}），耗時 {(time.time() - t0) / 60:.1f} 分鐘。")

with open(PROMPTS_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

finished_count = 0
for item in data.get("items", []):
    item.pop("_ref_image", None)
    out_file = os.path.join(INPUT_DIR, f"{item['name']}-front.png")
    if os.path.exists(out_file):
        item["done"] = True
        finished_count += 1

with open(PROMPTS_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

images = sorted(n for n in os.listdir(INPUT_DIR) if n.endswith(".png"))
archive_path = f"/kaggle/working/imagegen-{time.strftime('%Y%m%d-%H%M')}.zip"
with zipfile.ZipFile(archive_path, "w") as zf:
    for img_name in images:
        zf.write(os.path.join(INPUT_DIR, img_name), f"in/{img_name}")
    zf.write(PROMPTS_PATH, "prompts.json")

print(f"\n全部完成！共產出 {finished_count} 張正面圖，已打包至：{archive_path}")

In [ ]:
# Cell 5: Preview — reference (left) vs result (right) for each item
import glob, os
from PIL import Image
from IPython.display import display

fronts = sorted(glob.glob(os.path.join(INPUT_DIR, "*-front.png")))
if not fronts:
    print("尚無生成圖片。")
else:
    tile = 512
    sheet = Image.new("RGB", (tile * 2, tile * len(fronts)), (128, 128, 128))
    for i, path in enumerate(fronts):
        name = os.path.basename(path)[:-len("-front.png")]
        ref_path = os.path.join(DEBUG_DIR, f"{name}-ref.png")
        for col, p in enumerate((ref_path, path)):
            if not os.path.exists(p):
                continue
            img = Image.open(p).convert("RGBA")
            img.thumbnail((tile, tile), Image.Resampling.LANCZOS)
            sheet.paste(img, (col * tile + (tile - img.width) // 2, i * tile + (tile - img.height) // 2), img)
        print(f"[{i+1}] {name}：左＝參考圖（去背後）　右＝結果")
    display(sheet)